|                |   |
:----------------|---|
| **Nombre**     |Santiago Escutia Ríos   |
| **Fecha**      | 10/5/2026  |
| **Expediente** |757839   |

### Gini Impurity (Índice Gini)

Es la medida por defecto en algoritmos como CART. Mide la probabilidad de clasificar incorrectamente un elemento elegido al azar si se etiquetara según la distribución de clases en el nodo.
* Fórmula: $$G = 1 - \sum_{i=1}^{C} p_i^2$$  Donde $p_i$ es la proporción de elementos de la clase $i$ en el nodo.
* Conexión con las particiones: Un Gini de 0 indica pureza total. El algoritmo busca la partición que maximice la reducción del índice Gini (Gini Gain).

### Entropía (Entropy)

Proveniente de la teoría de la información, mide el nivel de incertidumbre o desorden en un conjunto de datos.
* Fórmula: $$H = -\sum_{i=1}^{C} p_i \log_2(p_i)$$ 
* Conexión con las particiones: Se utiliza para calcular la Ganancia de Información (Information Gain). El árbol elige la partición que produce la mayor disminución de entropía respecto al nodo padre.

### Log Loss (Pérdida Logarítmica)

Aunque se asocia más a menudo con modelos probabilísticos (como Regresión Logística o Redes Neuronales), en el contexto de árboles se utiliza principalmente en Gradient Boosting Decision Trees (GBDT).
Mide qué tan cerca está la probabilidad predicha del valor real (0 o 1). Penaliza fuertemente las predicciones que son seguras pero incorrectas.
* Fórmula: $$LogLoss = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$$

# ¿Cuál es la diferencia entre Entropía y Log Loss?

Aunque matemáticamente son primas hermanas (la fórmula de la entropía cruzada es idéntica a la función de costo Log Loss), su aplicación y enfoque difieren:
1. Enfoque de Medición:
    * La Entropía se calcula sobre la distribución interna de un nodo (qué tan mezcladas están las clases dentro del grupo).
    * El Log Loss se calcula comparando una predicción probabilística contra la etiqueta real.
2. Uso en Árboles:
    * La Entropía se usa para decidir dónde cortar el árbol (en la fase de construcción de un árbol simple como ID3 o C4.5).
    * El Log Loss se usa como la función de optimización en modelos de ensamble (como XGBoost), donde cada árbol nuevo intenta corregir el error (residuo) del anterior basándose en el gradiente de esta pérdida.
3. Escala: La entropía suele usar base 2 ($\log_2$) para medir en "bits", mientras que Log Loss suele usar el logaritmo natural ($\ln$).

## Ejemplo de partición con cada criterio de decisión

Se utiliza el dataset **Default** que contiene información sobre si un cliente incumplió su deuda de tarjeta de crédito (`default`), si es estudiante (`student`), su saldo promedio (`balance`) e ingreso (`income`).

Para ilustrar cada criterio se aplica la misma partición candidata: **`balance > 1800`**, y se evalúa qué tan puros quedan los nodos resultantes según Gini, Entropía y Log Loss.

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("Default.csv")

# Convertir Yes/No a números
le_default = LabelEncoder()
le_student = LabelEncoder()

df['default'] = le_default.fit_transform(df['default'])
df['student'] = le_student.fit_transform(df['student'])

X = df[['student', 'balance', 'income']]
y = df['default']

df.head()

,default,student,balance,income
0,0,0,729.526495,44361.625074
1,0,1,817.180407,12106.134700
2,0,0,1073.549164,31767.138950
3,0,0,529.250605,35704.493940
4,0,0,785.655883,38463.495880


### Partición candidata: `balance > 1800`

Dividimos el dataset en dos nodos hijos y evaluamos cada criterio paso a paso.

In [2]:
# Definir la partición
mascara = X['balance'] > 1800
izq = y[~mascara]   # balance <= 1800
der = y[mascara]    # balance > 1800

n_total = len(y)

print(f"Total de muestras:                    {n_total}")
print(f"Nodo izquierdo (balance ≤ 1800): n = {len(izq)}")
print(f"Nodo derecho   (balance > 1800): n = {len(der)}")

Total de muestras:                    10000
Nodo izquierdo (balance ≤ 1800): n = 9712
Nodo derecho   (balance > 1800): n = 288


### Criterio 1 — Gini

El Gini mide la probabilidad de clasificar mal un elemento elegido al azar. Un valor de 0 significa nodo puro; 0.5 es el máximo de impureza para clasificación binaria.

$$\text{Gini ponderado} = \frac{n_{izq}}{n} \cdot G_{izq} + \frac{n_{der}}{n} \cdot G_{der}$$

$$\text{Ganancia Gini} = G_{raíz} - \text{Gini ponderado}$$

In [3]:
def gini(y):
    clases = np.unique(y)
    return 1 - sum((np.sum(y == c) / len(y))**2 for c in clases)

gini_raiz = gini(y)
gini_izq  = gini(izq)
gini_der  = gini(der)
gini_pond = (len(izq) / n_total) * gini_izq + (len(der) / n_total) * gini_der
ganancia_gini = gini_raiz - gini_pond

print("=" * 50)
print("PARTICIÓN CON GINI  (balance > 1800)")
print("=" * 50)
print(f"  Gini raíz (antes de partir):      {gini_raiz:.4f}")
print(f"  Nodo izquierdo (balance ≤ 1800):  {gini_izq:.4f}  (n={len(izq)})")
print(f"  Nodo derecho   (balance > 1800):  {gini_der:.4f}  (n={len(der)})")
print(f"  Gini ponderado (después):         {gini_pond:.4f}")
print(f"  Ganancia Gini:                    {ganancia_gini:.4f}")
print()
print("  → El nodo derecho es mucho más impuro: contiene")
print("    la mayoría de los clientes que SÍ incumplieron.")

PARTICIÓN CON GINI  (balance > 1800)
  Gini raíz (antes de partir):      0.0644
  Nodo izquierdo (balance ≤ 1800):  0.0346  (n=9712)
  Nodo derecho   (balance > 1800):  0.4922  (n=288)
  Gini ponderado (después):         0.0478
  Ganancia Gini:                    0.0166

  → El nodo derecho es mucho más impuro: contiene
    la mayoría de los clientes que SÍ incumplieron.


### Criterio 2 — Entropía

La entropía mide la incertidumbre o desorden en un nodo. Una entropía de 0 indica nodo puro; el máximo para clasificación binaria es 1 bit.

$$\text{Entropía ponderada} = \frac{n_{izq}}{n} \cdot H_{izq} + \frac{n_{der}}{n} \cdot H_{der}$$

$$\text{Ganancia de Información} = H_{raíz} - \text{Entropía ponderada}$$

In [4]:
def entropia(y):
    clases = np.unique(y)
    return -sum(
        (np.sum(y == c) / len(y)) * np.log2(np.sum(y == c) / len(y))
        for c in clases if np.sum(y == c) > 0
    )

ent_raiz = entropia(y)
ent_izq  = entropia(izq)
ent_der  = entropia(der)
ent_pond = (len(izq) / n_total) * ent_izq + (len(der) / n_total) * ent_der
ganancia_info = ent_raiz - ent_pond

print("=" * 50)
print("PARTICIÓN CON ENTROPÍA  (balance > 1800)")
print("=" * 50)
print(f"  Entropía raíz (antes de partir):  {ent_raiz:.4f} bits")
print(f"  Nodo izquierdo (balance ≤ 1800):  {ent_izq:.4f} bits  (n={len(izq)})")
print(f"  Nodo derecho   (balance > 1800):  {ent_der:.4f} bits  (n={len(der)})")
print(f"  Entropía ponderada (después):     {ent_pond:.4f} bits")
print(f"  Ganancia de Información:          {ganancia_info:.4f} bits")
print()
print("  → La entropía del nodo derecho es alta: hay mucha")
print("    mezcla de clases en balances altos.")

PARTICIÓN CON ENTROPÍA  (balance > 1800)
  Entropía raíz (antes de partir):  0.2107 bits
  Nodo izquierdo (balance ≤ 1800):  0.1278 bits  (n=9712)
  Nodo derecho   (balance > 1800):  0.9887 bits  (n=288)
  Entropía ponderada (después):     0.1526 bits
  Ganancia de Información:          0.0581 bits

  → La entropía del nodo derecho es alta: hay mucha
    mezcla de clases en balances altos.


### Criterio 3 — Log Loss

El Log Loss por nodo se calcula asignando como predicción la proporción de la clase positiva en ese nodo. Penaliza fuertemente las predicciones confiadas pero incorrectas.

$$LL_{nodo} = -[p \ln(p) + (1-p) \ln(1-p)]$$

$$\text{Log Loss ponderado} = \frac{n_{izq}}{n} \cdot LL_{izq} + \frac{n_{der}}{n} \cdot LL_{der}$$

In [5]:
def logloss_nodo(y):
    if len(y) == 0:
        return 0
    p = np.clip(np.sum(y == 1) / len(y), 1e-10, 1 - 1e-10)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

ll_raiz = logloss_nodo(y)
ll_izq  = logloss_nodo(izq)
ll_der  = logloss_nodo(der)
ll_pond = (len(izq) / n_total) * ll_izq + (len(der) / n_total) * ll_der
reduccion_ll = ll_raiz - ll_pond

print("=" * 50)
print("PARTICIÓN CON LOG LOSS  (balance > 1800)")
print("=" * 50)
print(f"  Log Loss raíz (antes de partir):  {ll_raiz:.4f}")
print(f"  Nodo izquierdo (balance ≤ 1800):  {ll_izq:.4f}  (n={len(izq)})")
print(f"  Nodo derecho   (balance > 1800):  {ll_der:.4f}  (n={len(der)})")
print(f"  Log Loss ponderado (después):     {ll_pond:.4f}")
print(f"  Reducción de Log Loss:            {reduccion_ll:.4f}")
print()
print("  → El Log Loss penaliza más las predicciones erróneas")
print("    con alta confianza, útil en modelos de ensamble.")

PARTICIÓN CON LOG LOSS  (balance > 1800)
  Log Loss raíz (antes de partir):  0.1460
  Nodo izquierdo (balance ≤ 1800):  0.0886  (n=9712)
  Nodo derecho   (balance > 1800):  0.6853  (n=288)
  Log Loss ponderado (después):     0.1058
  Reducción de Log Loss:            0.0403

  → El Log Loss penaliza más las predicciones erróneas
    con alta confianza, útil en modelos de ensamble.


### Comparación final entre los tres criterios

In [6]:
print("=" * 55)
print("COMPARACIÓN FINAL — partición: balance > 1800")
print("=" * 55)
print(f"{'Criterio':<18} {'Valor raíz':>12} {'Valor post':>12} {'Ganancia':>10}")
print("-" * 55)
print(f"{'Gini':<18} {gini_raiz:>12.4f} {gini_pond:>12.4f} {ganancia_gini:>10.4f}")
print(f"{'Entropía (bits)':<18} {ent_raiz:>12.4f} {ent_pond:>12.4f} {ganancia_info:>10.4f}")
print(f"{'Log Loss':<18} {ll_raiz:>12.4f} {ll_pond:>12.4f} {reduccion_ll:>10.4f}")
print()
print("Conclusiones:")
print("  • Gini mide la probabilidad de clasificar mal un elemento al azar.")
print("  • Entropía mide la incertidumbre (en bits) del nodo.")
print("  • Log Loss mide el error de las probabilidades predichas.")
print("  • Los tres coinciden en que balance > 1800 es una buena partición:")
print("    todos producen una ganancia positiva al crear nodos más puros.")

COMPARACIÓN FINAL — partición: balance > 1800
Criterio             Valor raíz   Valor post   Ganancia
-------------------------------------------------------
Gini                     0.0644       0.0478     0.0166
Entropía (bits)          0.2107       0.1526     0.0581
Log Loss                 0.1460       0.1058     0.0403

Conclusiones:
  • Gini mide la probabilidad de clasificar mal un elemento al azar.
  • Entropía mide la incertidumbre (en bits) del nodo.
  • Log Loss mide el error de las probabilidades predichas.
  • Los tres coinciden en que balance > 1800 es una buena partición:
    todos producen una ganancia positiva al crear nodos más puros.


In [7]:
# Entrenamiento de árboles con sklearn para cada criterio
tree_gini = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=1)
tree_gini.fit(X, y)

tree_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=2, random_state=1)
tree_entropy.fit(X, y)

tree_log = DecisionTreeClassifier(criterion='log_loss', max_depth=2, random_state=1)
tree_log.fit(X, y)

probs = tree_log.predict_proba(X)
loss  = log_loss(y, probs)

print("========== IMPUREZA RAÍZ — SKLEARN ==========")
print(f"GINI:     {tree_gini.tree_.impurity[0]:.4f}")
print(f"ENTROPÍA: {tree_entropy.tree_.impurity[0]:.4f}")
print(f"LOG LOSS: {loss:.4f}")

========== IMPUREZA RAÍZ — SKLEARN ==========
GINI:     0.0644
ENTROPÍA: 0.2107
LOG LOSS: 0.0820


* GINI mide qué tan mezcladas están las clases.
* Entropía mide la incertidumbre.
* Log Loss mide qué tan equivocadas son las probabilidades predichas.
* El árbol escoge particiones que reduzcan estos valores y generen nodos más puros.